# Persona Multi-LoRA workshop: train one client adapter

**Primary question:** Can one independently trained LoRA adapter improve email replies for a selected client while keeping the shared base frozen?

**Time:** 25 minutes instruction plus hardware-dependent training.

Prerequisite: complete [04: persona response data](04_persona_response_data.ipynb) for the same `CLIENT_ID` and `PERSONA_RUN_ROOT`.


## Training contract

- Train one adapter for the selected client from the same pinned base model used by every other client.
- Start from a fresh base/LoRA/optimizer for each client; never continue another person's adapter.
- Apply loss only to the reviewed reply and EOS. Persona prompt tokens, incoming email tokens, and padding use `-100`.
- Shorten the incoming body before the response; never truncate the supervised reply.
- Select the best checkpoint by validation loss. The test partition is never passed to Trainer.
- Save safetensors, tokenizer, hashes, profile/data identity, and local Trackio metrics.
- Training is opt-in and never pushes to the Hub.


In [1]:
from __future__ import annotations

import os
import sys
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

import pandas as pd
from IPython.display import display

import email_workshop
from email_workshop.model_profiles import (
    DEFAULT_MODEL_PROFILE,
    model_profile_keys,
    resolve_model_profile,
)
from email_workshop.persona import (
    load_persona_dataset,
    load_persona_registry,
    persona_training_example,
    train_persona_adapter,
    verify_persona_adapter,
)
from email_workshop.training import TrainConfig

WORKSHOP_ROOT = Path(email_workshop.__file__).resolve().parents[1]
PERSONA_REGISTRY = load_persona_registry(WORKSHOP_ROOT)
CLIENT_ID = 'maya-chen-ceo'
INPUT_FILE = WORKSHOP_ROOT / PERSONA_REGISTRY[CLIENT_ID]['data_file']
PERSONA_RUN_ROOT = WORKSHOP_ROOT / Path(os.getenv('EMAIL_PERSONA_RUN', 'artifacts/persona-lora-run-2026'))
# Choose any key printed by model_profile_keys(), or use 'custom'.
MODEL_CHOICE = DEFAULT_MODEL_PROFILE
CUSTOM_MODEL_ID = None
CUSTOM_MODEL_REVISION = None
CUSTOM_TARGET_MODULES = None
MODEL = resolve_model_profile(
    MODEL_CHOICE,
    model_id=CUSTOM_MODEL_ID,
    revision=CUSTOM_MODEL_REVISION,
    target_modules=tuple(CUSTOM_TARGET_MODULES) if CUSTOM_TARGET_MODULES else None,
)
RUN_TOKENIZATION_CHECK = False
RUN_TRAINING = False
SMOKE_ONLY = True
DEVICE = 'cuda'
CONFIG = TrainConfig(
    model_profile=MODEL.key,
    model_id=MODEL.model_id,
    model_revision=MODEL.revision,
    target_modules=MODEL.target_modules,
    device=DEVICE,
    max_length=1024,
    max_steps=2 if SMOKE_ONLY else -1,
    sample_limit=16 if SMOKE_ONLY else None,
)
print(f'Kernel: {sys.executable}')
print(f"Client/adapter: {CLIENT_ID} -> {PERSONA_REGISTRY[CLIENT_ID]['adapter']}")
print(f'Model: {MODEL.key} -> {MODEL.model_id}@{MODEL.revision}')
print(f'Training enabled: {RUN_TRAINING}; smoke only: {SMOKE_ONLY}')
print(f'Available model choices: {model_profile_keys()}')


h:\masterswork\gitrash\multi-lora\workshops\email-multilora\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Kernel: h:\masterswork\gitrash\multi-lora\workshops\email-multilora\.venv\Scripts\python.exe
Client/adapter: maya-chen-ceo -> maya-chen
Model: qwen2.5-0.5b -> Qwen/Qwen2.5-0.5B-Instruct@7ae557604adf67be50417f59c2c2f167def9a775
Training enabled: False; smoke only: True
Available model choices: ('nemotron-nano-8b', 'llama3.1-8b', 'qwen2.5-7b', 'mistral-7b-v0.3', 'phi-4-mini', 'qwen2.5-3b', 'gemma2-2b', 'smollm2-1.7b', 'qwen2.5-0.5b', 'custom')


## Verify the training environment

Install the workshop `[training]` extra and select the `email-multilora` kernel. CUDA is recommended for a full run; CPU is intended only for a tiny smoke test. The 1,024-token sequence budget leaves room for the persona profile, incoming email, and complete response.


In [2]:
versions = {}
missing_packages = []
for package in ['torch', 'transformers', 'peft', 'accelerate', 'trackio']:
    try:
        versions[package] = version(package)
    except PackageNotFoundError:
        versions[package] = 'NOT INSTALLED'
        missing_packages.append(package)
display(pd.Series(versions))
if missing_packages:
    raise RuntimeError(
        f"Selected kernel {sys.executable} is missing {missing_packages}. "
        "In VS Code choose Select Kernel > Python Environments > "
        r"workshops\email-multilora\.venv\Scripts\python.exe, restart the kernel, "
        "and rerun from the configuration cell. Do not install into a parent workspace environment."
    )
if RUN_TRAINING or RUN_TOKENIZATION_CHECK:
    import torch
    print(f'CUDA available: {torch.cuda.is_available()}')
    if DEVICE == 'cuda' and not torch.cuda.is_available():
        raise RuntimeError('CUDA requested but unavailable.')


torch           2.10.0+cu128
transformers          4.57.6
peft                  0.18.1
accelerate            1.12.0
trackio                0.7.0
dtype: object

## Verify client and data lineage before spending compute

The adapter directory is `persona_adapters/<adapter-name>` inside the shared persona run. Reusing one run root lets vLLM load multiple client adapters later, but every adapter must use an identical base revision and training configuration.

Existing adapters are never overwritten. Choose a new artifact root when changing the model or hyperparameters.


In [3]:
splits, persona, profile_text, data_manifest = load_persona_dataset(
    WORKSHOP_ROOT,
    CLIENT_ID,
    INPUT_FILE,
)
adapter_dir = PERSONA_RUN_ROOT / 'persona_adapters' / persona['adapter']
display(pd.Series(data_manifest['split_counts']))
display(pd.Series(CONFIG.__dict__))
print(f'Adapter destination: {adapter_dir}')
if RUN_TRAINING and adapter_dir.exists():
    raise FileExistsError('The selected persona adapter already exists; use a new run or client.')


train         1473
validation     312
test           318
dtype: int64

model_profile                                        qwen2.5-0.5b
model_id                               Qwen/Qwen2.5-0.5B-Instruct
model_revision           7ae557604adf67be50417f59c2c2f167def9a775
target_modules                                   (q_proj, v_proj)
seed                                                         2026
max_length                                                   1024
rank                                                            8
epochs                                                        2.0
learning_rate                                              0.0002
batch_size                                                      1
gradient_accumulation                                           8
max_steps                                                       2
sample_limit                                                   16
device                                                       cuda
dtype: object

Adapter destination: H:\masterswork\gitrash\multi-lora\workshops\email-multilora\artifacts\persona-lora-run-2026\persona_adapters\maya-chen


## Inspect completion-only masking

Enable `RUN_TOKENIZATION_CHECK` to download the tokenizer and verify that only response tokens receive loss. The cell reports counts and does not print the incoming email or authored response.


In [4]:
if RUN_TOKENIZATION_CHECK:
    from transformers import AutoTokenizer
    tokenizer = AutoTokenizer.from_pretrained(CONFIG.model_id, revision=CONFIG.model_revision)
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token
    example = persona_training_example(
        tokenizer,
        splits['train'][0],
        persona,
        profile_text,
        CONFIG.max_length,
    )
    loss_tokens = [token for token in example['labels'] if token != -100]
    display(pd.Series({
        'total_tokens': len(example['input_ids']),
        'masked_prompt_tokens': sum(token == -100 for token in example['labels']),
        'response_loss_tokens_including_eos': len(loss_tokens),
        'fits_budget': len(example['input_ids']) <= CONFIG.max_length,
    }))
else:
    print('Tokenization inspection skipped; set RUN_TOKENIZATION_CHECK = True to enable.')


Tokenization inspection skipped; set RUN_TOKENIZATION_CHECK = True to enable.


## Run selected-client training explicitly

1. Keep `SMOKE_ONLY = True` for a two-step pipeline check in a fresh run root.
2. For a quality run, choose another root, set `SMOKE_ONLY = False`, and train the full train split.
3. Repeat this notebook with a different `CLIENT_ID` to add another independent adapter to the shared base.
4. Do not tune prompts or hyperparameters after viewing final test results.


In [ ]:
if RUN_TRAINING:
    result = train_persona_adapter(
        PERSONA_RUN_ROOT,
        WORKSHOP_ROOT,
        CLIENT_ID,
        CONFIG,
        INPUT_FILE,
    )
    display(pd.Series({
        'client_id': result['client_id'],
        'adapter': result['adapter'],
        'purpose': result['purpose'],
        'train_rows': result['train_rows'],
        'validation_rows': result['validation_rows'],
        'trainable_parameters': result['trainable_parameters'],
        'elapsed_seconds': result['elapsed_seconds'],
        'train_loss': result['metrics']['train_loss'],
    }))
else:
    print('Training skipped. Set RUN_TRAINING = True after reviewing the configuration.')


## Review the saved adapter identity

Validation loss measures next-token fit to reviewed replies; it does not prove that personality, correctness, or safety improved. Notebook 06 compares the frozen base and adapted model on the same held-out records and asks for human rubric review.


In [ ]:
manifest_path = adapter_dir / 'manifest.json'
if manifest_path.exists():
    manifest = verify_persona_adapter(PERSONA_RUN_ROOT, WORKSHOP_ROOT, CLIENT_ID)
    display(pd.Series({
        'client_id': manifest['client_id'],
        'adapter': manifest['adapter'],
        'purpose': manifest['purpose'],
        'data_sha256': manifest['data_manifest']['dataset_sha256'],
        'profile_sha256': manifest['data_manifest']['profile_sha256'],
        'best_checkpoint': manifest['best_checkpoint'],
    }))
else:
    print('No adapter manifest yet. Enable training or point to a completed persona run.')


## Ready for notebook 06 when

- The adapter manifest names the intended client and data/profile hashes.
- The base revision and LoRA targets are correct.
- A smoke adapter is treated only as pipeline evidence.
- The final test split remains sealed.
